# LightGBM (M3) tuning on Kaggle

Runs `heatwave tune-m3` (reference.md §4.6) on Kaggle instead of the laptop: more memory, and the laptop stays free. **All logic is in `src/heatwave`**; this notebook only installs the project, pulls the data, checks the GPU and runs the CLI.

**One-time setup (Kaggle notebook settings):**
1. Accelerator: **GPU T4 x2**. Internet: **on**.
2. Add-ons → Secrets: add **`HF_TOKEN`**, a Hugging Face token with **write** access to the private dataset `tarun-rai/cro-heatwave-data`, and attach it to this notebook.
3. Keep this notebook **private** (the data includes IMD-derived material that can't be shared).

**Run:** *Save Version → Save & Run All*, so it runs unattended (up to 12 h). After every trial the study is uploaded to Hugging Face; if the session ends early, run it again and it resumes. The 40-trial cap per target is fixed in `configs/models.yaml`.

**Afterwards, on the laptop:** `uv run heatwave hub-pull --only models`.

In [ ]:
REF = "main"  # branch or commit of github.com/tarun-rai21/CRO-Heatwave
TARGETS = ["lead", "window"]
THREADS = 4  # Kaggle sessions have 4 CPU cores

In [ ]:
import json
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
WORK = Path("/kaggle/working")

## 1. Install the project (exact locked versions)

In [ ]:
!pip install -q uv
!git clone -q https://github.com/tarun-rai21/CRO-Heatwave.git /kaggle/working/CRO-Heatwave || true
%cd /kaggle/working/CRO-Heatwave
!git fetch -q origin {REF} && git checkout -q --detach FETCH_HEAD && git log -1 --oneline
!uv sync --frozen --no-dev

## 2. Data: features, labels and the tuning study so far

In [ ]:
!uv run heatwave hub-pull --only features labels models
!du -sh data/*

## 3. GPU or CPU?

The default parameters are early-stopped once on fold 5 on each device. **The GPU is used only if it is at least 1.5x faster and its inner log loss is within 0.0001 of the CPU's** (trials differ by ~0.0005, so a bigger device difference could reorder them). Otherwise the CPU is used. Whichever is chosen must also be used for the final walk-forward and the final model.

In [ ]:
!nvidia-smi -L || echo "no GPU"
# LightGBM's GPU build uses OpenCL; register NVIDIA's driver with it if the image doesn't.
ICD = "/etc/OpenCL/vendors/nvidia.icd"
!mkdir -p /etc/OpenCL/vendors && (test -f {ICD} || echo libnvidia-opencl.so.1 > {ICD})
BENCH = f"uv run heatwave bench-m3 --target lead --fold 5 --threads {THREADS}"
!{BENCH} --devices cpu --out {WORK}/bench_cpu.json
!{BENCH} --devices gpu --out {WORK}/bench_gpu.json || echo "GPU run failed"

In [ ]:
cpu = json.loads((WORK / "bench_cpu.json").read_text())[0]
gpu_file = WORK / "bench_gpu.json"
gpu = json.loads(gpu_file.read_text())[0] if gpu_file.exists() else None
DEVICE = "cpu"
if gpu is not None:
    faster = cpu["seconds"] / gpu["seconds"]
    diff = abs(gpu["inner_logloss"] - cpu["inner_logloss"])
    print(f"GPU {faster:.2f}x faster; log-loss difference {diff:.6f}")
    print(f"rounds: gpu {gpu['rounds']}, cpu {cpu['rounds']}")
    if faster >= 1.5 and diff <= 1e-4:
        DEVICE = "gpu"
print("device:", DEVICE, "| cpu:", cpu, "| gpu:", gpu)

## 4. Tune (resumes from the study on Hugging Face; uploads after every trial)

In [ ]:
for target in TARGETS:  # noqa: B007 (used in the shell line)
    !uv run heatwave tune-m3 --target {target} --device {DEVICE} --threads {THREADS} --push

In [ ]:
for target in TARGETS:
    r = json.loads(Path(f"data/models/m3_tuning_{target}.json").read_text())
    print(target, "| trials:", r["trials"], "| code:", r["code_version"])
    for k in ("default", "best"):
        if r[k]:
            t = r[k]
            loss = t["mean_inner_logloss"]
            print(f"  {k}: trial {t['trial']}, inner log loss {loss:.5f}, rounds {t['rounds']}")